# 5주차 CNN 과제 (22기 금강산)

## 과제 요약
- **데이터셋**: Fashion-MNIST (CIFAR10 대신, 학습 시간·리소스 고려)
- **추가 기법**: Early Stopping, Dropout, Batch Normalization, stride/padding 변형, He 초기화, Adam + weight decay
- **분석**: 학습 곡선 및 검증 정확도 비교

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy
import numpy as np

## 1. 데이터셋: Fashion-MNIST
- CIFAR10 대신 **Fashion-MNIST** 사용 (28x28 그레이스케일, 10클래스, 학습이 빠르고 리소스 적음)
- 학습/검증 split, augmentation 적용

In [ ]:
# Transform: Normalize만 (Fashion-MNIST는 0~1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 데이터 로드 (5주차 폴더 기준 상위 data 폴더 사용)
data_path = '../data'
train_data = datasets.FashionMNIST(data_path, train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(data_path, train=False, download=True, transform=transform)

# 학습의 10%를 검증용으로 사용 (Early Stopping용)
n_train = len(train_data)
n_val = int(0.1 * n_train)
n_train = n_train - n_val
train_subset, val_subset = torch.utils.data.random_split(train_data, [n_train, n_val])

# 클래스 이름 (시각화용)
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# 샘플 시각화
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[label])
    ax.axis('off')
plt.suptitle('Fashion-MNIST 샘플')
plt.tight_layout()
plt.show()

## 2. 모델 설계 (변형 사항)
- **Conv 레이어**: stride=2 일부 적용해 다운샘플링, padding=1로 spatial 유지
- **Batch Normalization**: Conv 직후 적용
- **Dropout**: FC 직전/사이에 적용 (과적합 완화)
- **He 초기화**: Conv/Linear 가중치 초기화

In [ ]:
class BasicBlock(nn.Module):
    """Conv + BN + ReLU + (선택) stride로 다운샘플링"""
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, use_bn=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_ch) if use_bn else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x


class CNN(nn.Module):
    def __init__(self, num_classes=10, dropout_p=0.5):
        super().__init__()
        # Block1: 28x28 -> 14x14 (stride=2 in pool)
        self.block1 = nn.Sequential(
            BasicBlock(1, 32, kernel_size=3, stride=1, padding=1),
            BasicBlock(32, 32, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
        )
        # Block2: 14x14 -> 7x7
        self.block2 = nn.Sequential(
            BasicBlock(32, 64, kernel_size=3, stride=1, padding=1),
            BasicBlock(64, 64, kernel_size=3, stride=1, padding=1),
            nn.MaxPool2d(2, 2),
        )
        # Block3: 7x7 -> 3x3 (stride 2 conv로 축소)
        self.block3 = nn.Sequential(
            BasicBlock(64, 128, kernel_size=3, stride=2, padding=1),
            BasicBlock(128, 128, kernel_size=3, stride=1, padding=1),
            nn.AdaptiveAvgPool2d(1),
        )
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc1 = nn.Linear(128, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

## 3. Early Stopping & 학습 설정
- **Early Stopping**: 검증 손실이 `patience` 에폭 동안 개선되지 않으면 학습 중단
- **Optimizer**: Adam + weight decay (L2 regularization)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.counter = 0
        self.best_loss = None
        self.best_state = None
        self.stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

    def restore(self, model):
        if self.best_state is not None and self.restore_best_weights:
            model.load_state_dict(self.best_state)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 50
PATIENCE = 7

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = CNN(num_classes=10, dropout_p=0.5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=1e-4, restore_best_weights=True)

## 4. 학습 루프 (검증 + Early Stopping)

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += criterion(logits, y).item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(MAX_EPOCHS):
    model.train()
    running_loss, correct, n = 0.0, 0, 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{MAX_EPOCHS}', leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += x.size(0)

    train_loss = running_loss / n
    train_acc = correct / n
    val_loss, val_acc = evaluate(model, val_loader)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if early_stopping(val_loss, model):
        print(f"Early stopping at epoch {epoch+1}")
        break

early_stopping.restore(model)

## 5. 결과 분석
- 학습/검증 손실·정확도 곡선
- 테스트셋 최종 정확도

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(history['train_loss']) + 1)
axes[0].plot(epochs, history['train_loss'], label='Train Loss')
axes[0].plot(epochs, history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Loss curve')
axes[1].plot(epochs, history['train_acc'], label='Train Acc')
axes[1].plot(epochs, history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].set_title('Accuracy curve')
plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = evaluate(model, test_loader)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

## 6. 적용 기법 요약 및 분석

| 기법 | 적용 내용 | 효과 |
|------|-----------|------|
| **데이터셋** | Fashion-MNIST (28×28, 10클래스) | CIFAR10 대비 빠른 학습, 적은 리소스 |
| **Batch Normalization** | Conv 직후 BN | 내부 공변량 이동 완화, 학습 안정화 |
| **Dropout** | FC 전 0.5, fc1–fc2 사이 0.5 | 과적합 감소 |
| **Stride/Padding** | block3에서 stride=2로 7→3 축소, padding=1 유지 | 파라미터·연산량 절감, 공간 해상도 조절 |
| **He 초기화** | Conv/Linear에 Kaiming normal | ReLU에서 gradient 폭발/소실 완화 |
| **Weight decay** | Adam에 1e-4 | L2 정규화로 일반화 향상 |
| **Early Stopping** | 검증 손실 7에폭 개선 없으면 중단 | 과적합 방지, 최적 가중치 복원 |

- **Conv 레이어 구성**: BasicBlock을 3블록 사용 (32→64→128 채널), 마지막에 AdaptiveAvgPool2d(1)로 global pooling 후 FC.
- 위 설정으로 검증 정확도와 테스트 정확도를 함께 확인하면, 과적합 없이 일반화가 잘 되는지 판단할 수 있습니다.